In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------ CONFIG ------------------

benchmarks = [
    #"private_enterprise",
    #"social_media_cloud",
    #"commercial_cloud",
    "university"
]

#benchmarks = [
#    "private_enterprise",
#    "social_media_cloud"
#]

loads = range(7, 10)

link = 4

input_dir = "/home/hsd/workspace/trafpy/examples/comparison_generator/final_data"

# Existing ns3 outputs
output_dir = f"/home/hsd/workspace/ns3-load-balance/results_fix_asym/"

# RTT outputs (change this)
#rtt_dir = "/home/hsd/workspace/ns3-load-balance/results_rtt_fix"

new_rtt_dir = f"/home/hsd/workspace/ns3-load-balance/results_rtt_fix1_asym"

save_dir = f"/home/hsd/workspace/ns3-load-balance/results_rtt_fix1_asym/analysis_results"
Path(save_dir).mkdir(exist_ok=True)

ip_pattern = r'^\d+\.\d+\.\d+\.\d+$'

summary_results = {}

# ------------------ FUNCTIONS ------------------

def clean(df):
    return df[
        df['Src'].str.match(ip_pattern, na=False) &
        df['Dest'].str.match(ip_pattern, na=False)
    ].copy()


def ip_to_node(ip):
    last = int(ip.split('.')[-1])
    last1 = int(ip.split('.')[-2])

    leafcount = 2
    leaf = 0

    if last1 == 2:
        leaf = 1

    return ((last // 2) - 1 + (leaf * leafcount))


def add_time(df):

    df['start_time'] = (
        df['TimeFirstTxPacket']
        .astype(str)
        .str.replace('+', '', regex=False)
        .str.replace('ns', '', regex=False)
        .astype(float) / 1e9
    )

    return df


flow_columns = [
    "FlowID",
    "Src",
    "Dest",
    "TimeFirstRxPacket",
    "TimeFirstTxPacket",
    "TimeLastRxPacket",
    "TimeLastTxPacket",
    "FCT(s)",
    "TxPackets",
    "RxPackets",
    "LostPackets",
    "LossRate",
    "PDR",
    "LossPercent",
    "TxBytes",
    "RxBytes",
    "Throughput(Kbps)",
    "MeanDelay(ms)",
    "Jitter(ms)",
    "HopCount"
]


# ------------------ MATCH ------------------

def match_df(output_df, input_df, label):

    matches = []

    for _, in_row in input_df.iterrows():

        cand = output_df[
            (output_df.sn == in_row.sn) &
            (output_df.dn == in_row.dn)
        ].copy()

        if len(cand) == 0:
            continue

        cand['time_diff'] = abs(
            cand['start_time'] - in_row.event_time
        )

        best = cand.loc[cand['time_diff'].idxmin()]

        combined = {
            'flow_id': in_row.flow_id,
            'sn': in_row.sn,
            'dn': in_row.dn,
            'input_time': in_row.event_time,
            'flow_size': in_row.flow_size,

            f'time_diff_{label}': best['time_diff'],

            f'TxPackets_{label}': best['TxPackets'],
            f'RxPackets_{label}': best['RxPackets'],

            f'TxBytes_calc_{label}': best['TxPackets'] * 1400,
            f'RxBytes_calc_{label}': best['RxPackets'] * 1400,
        }

        for col in flow_columns:
            combined[f"{col}_{label}"] = best[col]

        matches.append(combined)

    return pd.DataFrame(matches)


# ------------------ MAIN ------------------

for benchmark in benchmarks:

    for load in loads:

        print(f"\n=== {benchmark} | Load 0.{load} ===")

        input_path = (
            f"{input_dir}/{benchmark}_load_{load}.csv"
        )

        try:

            input_df = pd.read_csv(input_path)

            timeout_values = ["true", "false"]

            algorithms = {
                "conga": pd.read_csv(
                    f"{output_dir}/{benchmark}_load_{load}_Conga.csv",
                    nrows=12000
                ),
                
                "conga_new": pd.read_csv(
                    f"{output_dir}/{benchmark}_load_{load}_Conga_new.csv",
                    nrows=12000
                ),

                "conga-ecmp": pd.read_csv(
                    f"{output_dir}/{benchmark}_load_{load}_Conga-ECMP_new.csv",
                    nrows=12000
                ),

                "ecmp": pd.read_csv(
                    f"{output_dir}/{benchmark}_load_{load}_ECMP.csv",
                    nrows=12000
                ),

                "ecmp_new": pd.read_csv(
                    f"{output_dir}/{benchmark}_load_{load}_ECMP_new.csv",
                    nrows=12000
                ),
            }

            for timeout in timeout_values:
            
                algorithms[f"weighted_{timeout}"] = pd.read_csv(
                    f"{new_rtt_dir}/{benchmark}_load_{load}_WEIGHTED_ECMP_timeout_{timeout}_RTT.csv",
                    nrows=12000
                )

                algorithms[f"random_{timeout}"] = pd.read_csv(
                    f"{new_rtt_dir}/{benchmark}_load_{load}_POWER_OF_2_RANDOM_timeout_{timeout}_RTT.csv",
                    nrows=12000
                )

                algorithms[f"top2_{timeout}"] = pd.read_csv(
                    f"{new_rtt_dir}/{benchmark}_load_{load}_POWER_OF_2_TOP2_timeout_{timeout}_RTT.csv",
                    nrows=12000
                )

        except Exception as e:
            print("Skipping:", e)
            continue

        # ------------------ CLEAN ------------------

        for label, df in algorithms.items():

            df = add_time(clean(df))

            df['sn'] = df['Src'].apply(ip_to_node)
            df['dn'] = df['Dest'].apply(ip_to_node)

            algorithms[label] = df

        # ------------------ MATCH ------------------

        matched = {}

        for label, df in algorithms.items():
            matched[label] = match_df(df, input_df, label)

        merge_keys = [
            'flow_id',
            'sn',
            'dn',
            'input_time',
            'flow_size'
        ]

        algorithms_list = list(algorithms.keys())

        final_df = matched[algorithms_list[0]]

        for algo in algorithms_list[1:]:
            final_df = pd.merge(
                final_df,
                matched[algo],
                on=merge_keys,
                how='inner'
            )

        if len(final_df) == 0:
            print("No matches.")
            continue


        # ------------------ SAVE CSV ------------------

        csv_path = (
            f"{save_dir}/{benchmark}_load_{load}_full.csv"
        )

        final_df.to_csv(csv_path, index=False)

        # ------------------ STATS ------------------

        stats = {}

        for algo in algorithms:
            fct = final_df[f"FCT(s)_{algo}"]

            stats[algo] = {
                "avg": fct.mean(),
                "p75": fct.quantile(0.75),
                "p90": fct.quantile(0.90),
                "p95": fct.quantile(0.95),
                "p99": fct.quantile(0.99),
            }

        
        for algo in stats:
            print(
                algo,
                "Avg:", stats[algo]["avg"],
                "P75:", stats[algo]["p75"],
                "P90:", stats[algo]["p90"],
                "P95:", stats[algo]["p95"],
                "P99:", stats[algo]["p99"],
            )

        if benchmark not in summary_results:

            summary_results[benchmark] = {
                "loads": []
            }

            for algo in algorithms:
                summary_results[benchmark][algo] = {
                "avg": [],
                "p75": [],
                "p90": [],
                "p95": [],
                "p99": []
                }

        summary_results[benchmark]["loads"].append(load / 10)

        for algo in algorithms:
            for metric in ["avg", "p75", "p90", "p95", "p99"]:
                summary_results[benchmark][algo][metric].append(
                    stats[algo][metric]
                )
        

# ------------------ FINAL SUMMARY PLOTS ------------------

metrics = ["avg", "p75", "p90", "p95", "p99"]

metric_titles = {
    "avg": "Average FCT",
    "p75": "75th Percentile FCT",
    "p90": "90th Percentile FCT",
    "p95": "95th Percentile FCT",
    "p99": "99th Percentile FCT"
}

marker_map = {
    "conga": "o",
    "conga_new": "h",
    "conga-ecmp": "p",
    "ecmp": "s",
    "ecmp_new": "*",

    "weighted_true": "D",
    "weighted_false": "d",

    "random_true": "^",
    "random_false": "v",

    "top2_true": "P",
    "top2_false": "X",
}

color_map = {
    "conga": "tab:blue",
    "conga_new": "tab:cyan",
    "conga-ecmp": "tab:green",
    "ecmp": "tab:red",
    "ecmp_new": "tab:purple",

    "weighted_true": "tab:brown",
    "weighted_false": "tab:brown",

    "random_true": "tab:orange",
    "random_false": "tab:orange",

    "top2_true": "tab:gray",
    "top2_false": "tab:gray",
}

label_map = {
    "conga": "Conga",
    "conga_new": "Conga_New",
    "conga-ecmp": "Conga-ECMP",
    "ecmp": "ECMP",
    "ecmp_new": "ECMP_New",

    "weighted_true": "Weighted ECMP (Timeout)",
    "weighted_false": "Weighted ECMP (No Timeout)",

    "random_true": "Power-of-2 Random (Timeout)",
    "random_false": "Power-of-2 Random (No Timeout)",

    "top2_true": "Power-of-2 Top2 (Timeout)",
    "top2_false": "Power-of-2 Top2 (No Timeout)",
}

global_limits = {}

for metric in metrics:
    values = []

    for benchmark in summary_results:
        for algo in summary_results[benchmark]:
            if algo == "loads":
                continue
            values.extend(summary_results[benchmark][algo][metric])

    ymin = min(values)
    ymax = max(values)
    margin = 0.05 * (ymax - ymin)

    global_limits[metric] = (ymin - margin, ymax + margin)

    

for benchmark in summary_results:

    loads_x = summary_results[benchmark]["loads"]

    for metric in metrics:

        plt.figure(figsize=(8, 5))

        for algo in [k for k in summary_results[benchmark] if k != "loads"]:

            linestyle = "--" if algo.endswith("_false") else "-"

            plt.plot(
                loads_x,
                summary_results[benchmark][algo][metric],
                marker=marker_map.get(algo, "o"),
                color=color_map.get(algo),
                linestyle=linestyle,
                
                linewidth=2,
                label=label_map.get(algo, algo.upper())
            )

        plt.xlabel("Load")
        plt.ylabel(metric_titles[metric])
        plt.title(f"{benchmark} {metric_titles[metric]} vs Load")
        plt.ylim(global_limits[metric])

        plt.grid(True)
        plt.legend()

        plt.tight_layout()

        plt.savefig(
            f"{save_dir}/{benchmark}_{metric}_vs_load.png",
            dpi=300
        )

        plt.close()

print("\nDONE: All comparison plots generated!")


=== university | Load 0.7 ===
conga Avg: 0.016615375833333335 P75: 0.022112 P90: 0.037953100000000003 P95: 0.051416350000000034 P99: 0.10242499000000006
conga_new Avg: 0.017203652666666666 P75: 0.02181325 P90: 0.039325500000000006 P95: 0.05545785 P99: 0.11214973000000003
conga-ecmp Avg: 4.012401505000001 P75: 7.5994465 P90: 12.9255032 P95: 15.975268 P99: 22.266234120000032
ecmp Avg: 0.16737123499999998 P75: 0.02934075 P90: 0.06953089999999999 P95: 0.11443475000000011 P99: 10.452942330000042
ecmp_new Avg: 0.06909265416666667 P75: 0.038034 P90: 0.08532280000000005 P95: 0.12373055000000004 P99: 0.24704198000000072
weighted_true Avg: 0.016722044166666665 P75: 0.0202635 P90: 0.03751810000000002 P95: 0.0528897 P99: 0.11042664000000027
random_true Avg: 0.0161797385 P75: 0.0194785 P90: 0.03667020000000002 P95: 0.05292465000000002 P99: 0.10945712000000035
top2_true Avg: 0.016722044166666665 P75: 0.0202635 P90: 0.03751810000000002 P95: 0.0528897 P99: 0.11042664000000027
weighted_false Avg: 0.01

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# ------------------ CONFIG ------------------

save_dir = Path(
    "/home/hsd/workspace/ns3-load-balance/results_rtt_fix1_asym/analysis_results"
)

graph_dir = save_dir / "graph"
graph_dir.mkdir(parents=True, exist_ok=True)

benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]

loads = range(1, 10)

SMALL = 100 * 1024        # 100 KB
LARGE = 1 * 1024 * 1024   # 1 MB

algorithms = [
    "conga",
    "conga_new",
    #"conga-ecmp",
    "ecmp",
    "ecmp_new",

    "weighted_true",
    "weighted_false",

    "random_true",
    "random_false",

    "top2_true",
    "top2_false"
]

label_map = {
    "conga": "Conga",
    "conga_new": "Conga_New",
    "conga-ecmp": "Conga-ECMP",
    "ecmp": "ECMP",
    "ecmp_new": "ECMP_New",

    "weighted_true": "Weighted ECMP (Timeout)",
    "weighted_false": "Weighted ECMP (No Timeout)",

    "random_true": "Power-of-2 Random (Timeout)",
    "random_false": "Power-of-2 Random (No Timeout)",

    "top2_true": "Power-of-2 Top2 (Timeout)",
    "top2_false": "Power-of-2 Top2 (No Timeout)"
}

color_map = {
    "conga": "tab:blue",
    "conga_new": "tab:cyan",
    "conga-ecmp": "tab:green",
    "ecmp": "tab:red",
    "ecmp_new": "tab:purple",

    "weighted_true": "tab:brown",
    "weighted_false": "tab:brown",

    "random_true": "tab:orange",
    "random_false": "tab:orange",

    "top2_true": "tab:gray",
    "top2_false": "tab:gray",
}

marker_map = {
    "conga": "o",
    "conga_new": "h",
    "conga-ecmp": "p",
    "ecmp": "s",
    "ecmp_new": "*",

    "weighted_true": "D",
    "weighted_false": "d",

    "random_true": "^",
    "random_false": "v",

    "top2_true": "P",
    "top2_false": "X"
}



# ------------------ ANALYSIS ------------------

for benchmark in benchmarks:

    print(f"\n===== {benchmark} =====")

    load_vals = []

    small_results = {algo: [] for algo in algorithms}
    large_results = {algo: [] for algo in algorithms}

    for load in loads:

        csv_path = save_dir / f"{benchmark}_load_{load}_full.csv"

        if not csv_path.exists():
            print(f"Missing: {csv_path}")
            continue

        df = pd.read_csv(csv_path)

        if df.empty:
            continue

        small_df = df[df["flow_size"] < SMALL]
        large_df = df[df["flow_size"] > LARGE]

        # ---------- Small Flows ----------

        for algo in algorithms:
        
            col = f"FCT(s)_{algo}"
        
            if col not in df.columns:
                small_results[algo].append(np.nan)
                large_results[algo].append(np.nan)
                continue
        
            # Small flows
            if not small_df.empty:
                small_results[algo].append(
                    small_df[col].mean()
                )
            else:
                small_results[algo].append(np.nan)
        
            # Large flows
            if not large_df.empty:
                large_results[algo].append(
                    large_df[col].mean()
                )
            else:
                large_results[algo].append(np.nan)

        load_vals.append(load / 10)

    # ==================================================
    # SMALL FLOWS
    # ==================================================

    plt.figure(figsize=(8, 5))

    for algo in algorithms:

        plt.plot(
            load_vals,
            small_results[algo],
            marker=marker_map[algo],
            color=color_map[algo],
            linestyle="--" if algo.endswith("_false") else "-",
            linewidth=2,
            label=label_map[algo]
        )

    plt.xlabel("Network Load")
    plt.ylabel("Average FCT (s)")
    plt.title(f"{benchmark}: Small Flows (<100 KB)")
    # plt.ylim(ylim)
    plt.grid(True)
    plt.legend()

    out_path = graph_dir / f"{benchmark}_SMALL_vs_load.png"

    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()

    print(f"Saved: {out_path}")

    # ==================================================
    # LARGE FLOWS
    # ==================================================

    plt.figure(figsize=(8, 5))

    for algo in algorithms:

        plt.plot(
            load_vals,
            large_results[algo],
            marker=marker_map[algo],
            color=color_map[algo],
            linestyle="--" if algo.endswith("_false") else "-",
            linewidth=2,
            label=label_map[algo]
        )

    plt.xlabel("Network Load")
    plt.ylabel("Average FCT (s)")
    plt.title(f"{benchmark}: Large Flows (>1 MB)")
    # plt.ylim(ylim)
    plt.grid(True)
    plt.legend()

    out_path = graph_dir / f"{benchmark}_LARGE_vs_load.png"

    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()

    print(f"Saved: {out_path}")

print("\nDone: Small vs Large flow comparison for all algorithms.")


===== private_enterprise =====
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix1_asym/analysis_results/graph/private_enterprise_SMALL_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix1_asym/analysis_results/graph/private_enterprise_LARGE_vs_load.png

===== social_media_cloud =====
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix1_asym/analysis_results/graph/social_media_cloud_SMALL_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix1_asym/analysis_results/graph/social_media_cloud_LARGE_vs_load.png

===== commercial_cloud =====
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix1_asym/analysis_results/graph/commercial_cloud_SMALL_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix1_asym/analysis_results/graph/commercial_cloud_LARGE_vs_load.png

===== university =====
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix1_asym/analysis_results/graph/university_SMALL_vs_load.png
Saved: /home

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
from itertools import combinations

# ------------------ CONFIG ------------------

save_dir = Path(
    "/home/hsd/workspace/ns3-load-balance/results_rtt_fix1_asym/analysis_results"
)

graph_dir = save_dir / "graph"
graph_dir.mkdir(parents=True, exist_ok=True)

benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]

loads = range(1, 10)

SMALL = 100 * 1024       # 100 KB
LARGE = 1 * 1024 * 1024  # 1 MB

# ------------------ UPDATED (8 CURVES) ------------------

algorithms = [
    "conga",
    "conga_new",
    "conga-ecmp",
    "ecmp",
    "ecmp_new",

    "weighted_true",
    "weighted_false",

    "random_true",
    "random_false",

    "top2_true",
    "top2_false"
]

label_map = {
    "conga": "Conga",
    "conga_new": "Conga_New",
    "conga-ecmp": "Conga-ECMP",
    "ecmp": "ECMP",
    "ecmp_new": "ECMP_New",

    "weighted_true": "Weighted ECMP (Timeout)",
    "weighted_false": "Weighted ECMP (No Timeout)",

    "random_true": "Power-of-2 Random (Timeout)",
    "random_false": "Power-of-2 Random (No Timeout)",

    "top2_true": "Power-of-2 Top2 (Timeout)",
    "top2_false": "Power-of-2 Top2 (No Timeout)"
}

color_map = {
    "conga": "tab:blue",
    "conga_new": "tab:cyan",
    "conga-ecmp": "tab:green",
    "ecmp": "tab:red",
    "ecmp_new": "tab:purple",

    "weighted_true": "tab:brown",
    "weighted_false": "tab:brown",

    "random_true": "tab:orange",
    "random_false": "tab:orange",

    "top2_true": "tab:gray",
    "top2_false": "tab:gray",
}

marker_map = {
    "conga": "o",
    "conga_new": "h",
    "conga-ecmp": "p",
    "ecmp": "s",
    "ecmp_new": "*",

    "weighted_true": "D",
    "weighted_false": "d",

    "random_true": "^",
    "random_false": "v",

    "top2_true": "P",
    "top2_false": "X"
}


# ------------------ MAIN ------------------

for benchmark in benchmarks:

    print(f"\n===== {benchmark} =====")

    load_vals = []

    small_results = {algo: [] for algo in algorithms}
    large_results = {algo: [] for algo in algorithms}

    for load in loads:

        csv_path = save_dir / f"{benchmark}_load_{load}_full.csv"

        if not csv_path.exists():
            print(f"Missing: {csv_path}")
            continue

        df = pd.read_csv(csv_path)

        if df.empty:
            continue

        # ------------------ FLOW SPLIT ------------------

        small_df = df[df["flow_size"] < SMALL]
        large_df = df[df["flow_size"] > LARGE]
        mid_df = ~(df["flow_size"] < SMALL) & ~(df["flow_size"] > LARGE)

        # ------------------ METRICS ------------------

        for algo in algorithms:

            col = f"FCT(s)_{algo}"

            if col not in df.columns:
                small_results[algo].append(np.nan)
                large_results[algo].append(np.nan)
                continue

            # small flows
            if not small_df.empty:
                small_results[algo].append(small_df[col].mean())
            else:
                small_results[algo].append(np.nan)

            # large flows
            if not large_df.empty:
                large_results[algo].append(large_df[col].mean())
            else:
                large_results[algo].append(np.nan)

        load_vals.append(load / 10)

        # ------------------ PAIRWISE SCATTER ------------------

        for a, b in combinations(algorithms, 2):

            col_a = f"FCT(s)_{a}"
            col_b = f"FCT(s)_{b}"

            if col_a not in df.columns or col_b not in df.columns:
                continue

            plt.figure(figsize=(7, 7))

            plt.scatter(
                df.loc[df["flow_size"] < SMALL, col_a],
                df.loc[df["flow_size"] < SMALL, col_b],
                alpha=0.5,
                label="Small (<100 KB)"
            )

            plt.scatter(
                df.loc[(df["flow_size"] >= SMALL) & (df["flow_size"] <= LARGE), col_a],
                df.loc[(df["flow_size"] >= SMALL) & (df["flow_size"] <= LARGE), col_b],
                alpha=0.5,
                label="Medium (100 KB - 1 MB)"
            )

            plt.scatter(
                df.loc[df["flow_size"] > LARGE, col_a],
                df.loc[df["flow_size"] > LARGE, col_b],
                alpha=0.5,
                label="Large (>1 MB)"
            )

            max_val = max(
                df[col_a].max(),
                df[col_b].max()
            )

            plt.plot(
                [0, max_val],
                [0, max_val],
                "--",
                color="black",
                label="y=x"
            )

            plt.xlabel(f"{label_map[a]} FCT (s)")
            plt.ylabel(f"{label_map[b]} FCT (s)")
            plt.title(
                f"{benchmark} Load 0.{load}\n"
                f"{label_map[b]} vs {label_map[a]}"
            )

            plt.grid(True)
            plt.legend()

            out_file = (
                graph_dir /
                f"{benchmark}_load_{load}_{a}_{b}_colored_scatter.png"
            )

            plt.tight_layout()
            plt.savefig(out_file, dpi=300)
            plt.close()

            print(f"Saved: {out_file}")

print("\nDone: Pairwise colored scatter plots generated for all algorithms.")


===== private_enterprise =====
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix1_asym/analysis_results/graph/private_enterprise_load_1_conga_ecmp_colored_scatter.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix1_asym/analysis_results/graph/private_enterprise_load_1_conga_ecmp_new_colored_scatter.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix1_asym/analysis_results/graph/private_enterprise_load_1_conga_weighted_true_colored_scatter.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix1_asym/analysis_results/graph/private_enterprise_load_1_conga_weighted_false_colored_scatter.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix1_asym/analysis_results/graph/private_enterprise_load_1_conga_random_true_colored_scatter.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix1_asym/analysis_results/graph/private_enterprise_load_1_conga_random_false_colored_scatter.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rt

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# ------------------ CONFIG ------------------

save_dir = Path(
    "/home/hsd/workspace/ns3-load-balance/results_rtt_fix1_asym/analysis_results"
)

baseline = "ecmp_new"

graph_dir = save_dir / f"graph/fct/normalised_vs_{baseline}"
graph_dir.mkdir(parents=True, exist_ok=True)


benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]

loads = range(1, 10)

SMALL = 100 * 1024
LARGE = 1 * 1024 * 1024



# ------------------ UPDATED (8 CURVES) ------------------

algorithms = [
    "conga",
    "conga_new",
    #"conga-ecmp",
    "ecmp",
    "ecmp_new",

    "weighted_true",
    "weighted_false",

    "random_true",
    "random_false",

    "top2_true",
    "top2_false"
]

label_map = {
    "conga": "Conga",
    "conga_new": "Conga_New",
    "conga-ecmp": "Conga-ECMP",
    "ecmp": "ECMP",
    "ecmp_new": "ECMP_New",

    "weighted_true": "Weighted ECMP (Timeout)",
    "weighted_false": "Weighted ECMP (No Timeout)",

    "random_true": "Power-of-2 Random (Timeout)",
    "random_false": "Power-of-2 Random (No Timeout)",

    "top2_true": "Power-of-2 Top2 (Timeout)",
    "top2_false": "Power-of-2 Top2 (No Timeout)"
}

color_map = {
    "conga": "tab:blue",
    "conga_new": "tab:cyan",
    "conga-ecmp": "tab:green",
    "ecmp": "tab:red",
    "ecmp_new": "tab:purple",

    "weighted_true": "tab:brown",
    "weighted_false": "tab:brown",

    "random_true": "tab:orange",
    "random_false": "tab:orange",

    "top2_true": "tab:gray",
    "top2_false": "tab:gray",
}

marker_map = {
    "conga": "o",
    "conga_new": "h",
    "conga-ecmp": "p",
    "ecmp": "s",
    "ecmp_new": "*",

    "weighted_true": "D",
    "weighted_false": "d",

    "random_true": "^",
    "random_false": "v",

    "top2_true": "P",
    "top2_false": "X"
}

# =========================================================
# MAIN
# =========================================================

for benchmark in benchmarks:

    print(f"\n===== {benchmark} =====")

    load_vals = []

    small_results = {algo: [] for algo in algorithms}
    large_results = {algo: [] for algo in algorithms}

    for load in loads:

        csv_path = save_dir / f"{benchmark}_load_{load}_full.csv"

        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path)

        if df.empty:
            continue

        # -------------------------------------------------
        # Avoid divide-by-zero
        # -------------------------------------------------

        base_col = f"FCT(s)_{baseline}"
        if base_col not in df.columns:
            continue

        df = df[df[base_col] > 0]

        if df.empty:
            continue

        # -------------------------------------------------
        # NORMALIZED FCT
        # -------------------------------------------------

        #for algo in algorithms:

         #   col = f"FCT(s)_{algo}"

          #  if col not in df.columns:
           #     df[f"{algo}_vs_{baseline}"] = np.nan
            #    continue

            #df[f"{algo}_vs_{baseline}"] = (
             #   df[col] / df[base_col]
            #)

        # -------------------------------------------------
        # FLOW SIZE SPLIT
        # -------------------------------------------------

        small_df = df[df["flow_size"] < SMALL]
        large_df = df[df["flow_size"] > LARGE]

        #for algo in algorithms:

         #   col = f"{algo}_vs_{baseline}"

          #  if not small_df.empty:
           #     small_results[algo].append(small_df[col].mean())
            #else:
             #   small_results[algo].append(np.nan)

            #if not large_df.empty:
             #   large_results[algo].append(large_df[col].mean())
            #else:
             #   large_results[algo].append(np.nan)

        for algo in algorithms:

            algo_col = f"FCT(s)_{algo}"

            # Skip if algorithm column is missing
            if algo_col not in df.columns:
                small_results[algo].append(np.nan)
                large_results[algo].append(np.nan)
                continue
        
            # ----- Small flows -----
            if not small_df.empty:
                algo_mean = small_df[algo_col].mean()
                baseline_mean = small_df[base_col].mean()
        
                small_results[algo].append(
                    algo_mean / baseline_mean if baseline_mean > 0 else np.nan
                )
            else:
                small_results[algo].append(np.nan)
        
            # ----- Large flows -----
            if not large_df.empty:
                algo_mean = large_df[algo_col].mean()
                baseline_mean = large_df[base_col].mean()
        
                large_results[algo].append(
                    algo_mean / baseline_mean if baseline_mean > 0 else np.nan
                )
            else:
                large_results[algo].append(np.nan)

        load_vals.append(load / 10)

    # =====================================================
    # SMALL FLOWS
    # =====================================================

    plt.figure(figsize=(8, 6), dpi=120)

    for algo in algorithms:

        plt.plot(
            load_vals,
            small_results[algo],
            marker=marker_map[algo],
            color=color_map.get(algo),
            linestyle="--" if algo.endswith("_false") else "-",
            linewidth=2,
            label=f"{label_map[algo]} / ECMP_New"
        )

    plt.axhline(
        y=1,
        linestyle="--",
        color="black"
    )

    plt.xlabel("Network Load")
    plt.ylabel("Normalized FCT (vs ECMP_New)")
    plt.title(f"{benchmark}: Small Flows (<100 KB)")

    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        graph_dir / f"{benchmark}_SMALL_relative_vs_load.png",
        dpi=300
    )

    plt.close()

    # =====================================================
    # LARGE FLOWS
    # =====================================================

    plt.figure(figsize=(8, 6), dpi=120)

    for algo in algorithms:

        plt.plot(
            load_vals,
            large_results[algo],
            marker=marker_map[algo],
            color=color_map.get(algo),
            linestyle="--" if algo.endswith("_false") else "-",
            linewidth=2,
            label=f"{label_map[algo]} / ECMP_New"
        )

    plt.axhline(
        y=1,
        linestyle="--",
        color="black"
    )

    plt.xlabel("Network Load")
    plt.ylabel("Normalized FCT (vs ECMP_New)")
    plt.title(f"{benchmark}: Large Flows (>1 MB)")

    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        graph_dir / f"{benchmark}_LARGE_relative_vs_load.png",
        dpi=300
    )

    plt.close()

    print(f"Finished {benchmark}")

print("\nDone: Normalized FCT graphs generated.")


===== private_enterprise =====
Finished private_enterprise

===== social_media_cloud =====
Finished social_media_cloud

===== commercial_cloud =====
Finished commercial_cloud

===== university =====
Finished university

Done: Normalized FCT graphs generated.
